# Setup

In [1]:
!pip install opik pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import json
import os
import re
from opik import Opik
from opik.evaluation.metrics import base_metric, score_result
from opik.integrations.openai import track_openai
from opik.evaluation import evaluate
from openai import OpenAI
from typing import Any

# Opik Configuration
os.environ["OPIK_URL_OVERRIDE"] = "https://3.110.54.210/api"
os.environ["OPIK_CHECK_TLS_CERTIFICATE"] = "false"
if not os.environ.get("OPENROUTER_API_KEY"):
    raise ValueError("Set OPENROUTER_API_KEY in your environment before running this notebook")
os.environ["OPIK_PROJECT_NAME"] = "AI Evaluations"

# Experiment Configuration
EXPERIMENT_NAME = "web-share-eval_google/openai/gpt-5.4-mini"
MODEL_NAME = "openai/gpt-5.4-mini"
DATASET_NAME = "ai_eval_web_share_prod"

# OpenRouter client
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# Opik client
opik_client = Opik(project_name=os.environ["OPIK_PROJECT_NAME"])

print("✅ Environment configured")

✅ Environment configured


# Upload Dataset

In [3]:
dataset = opik_client.get_or_create_dataset(DATASET_NAME)
print(dataset)

# Fetch all items (if small) and count
items = dataset.get_items()
current_size = len(items)

if current_size == 0:
    df = pd.read_csv("your_dataset.csv")
    dataset.insert_from_pandas(dataframe=df)
    print(f"✅ Uploaded {len(df)} rows to '{DATASET_NAME}'")
else:
    print(f"✅ Dataset '{DATASET_NAME}' already exists with {current_size} rows, skipping upload")

✅ Dataset 'ai_eval_web_share_prod' already exists with 97 rows, skipping upload


# Set Up Model Config

In [4]:
# ── All metrics use this shared evaluator LLM ──────────────────────────────
EVALUATOR_MODEL = "google/gemini-2.5-pro"
def call_evaluator(prompt: str) -> dict:
    response = openrouter_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.001,
    )
    raw = response.choices[0].message.content.strip()
    raw = re.sub(r"```json|```", "", raw).strip()
    return json.loads(raw)

print(f"✅ Evaluator LLM configured: {EVALUATOR_MODEL}")

✅ Evaluator LLM configured: google/gemini-2.5-pro


# Create your custom evaluators


In [5]:
def _safe_str(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and pd.isna(value):
        return ""
    return str(value)


def build_web_share_prompt(dataset_item: dict) -> str:
    prompt = dataset_item.get("prompt", "")
    variables = {
        "question_description": _safe_str(dataset_item.get("question_description")),
        "html_code": _safe_str(dataset_item.get("html_code")),
        "css_code": _safe_str(dataset_item.get("css_code")),
        "js_code": _safe_str(dataset_item.get("js_code")),
        "live_url": _safe_str(dataset_item.get("live_url")),
        "social_media": _safe_str(dataset_item.get("social_media")),
    }
    for key, value in variables.items():
        prompt = prompt.replace("{{" + key + "}}", value)
    return prompt

print("✅ Web share helpers defined")


✅ Web share helpers defined


# CodeImplementationAccuracyMetric


In [6]:
class CodeImplementationAccuracyMetric(base_metric.BaseMetric):
    """
    Scores 1 if every described feature, technology, and behavior is verifiably present in USER_CODE.
    Scores 0 if the post includes even one feature, technology, or behavior not present in USER_CODE.
    """
    def __init__(self, name: str = "code_implementation_accuracy"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the post strictly reflects features, technologies, and behaviors that are actually implemented in the provided USER_CODE.

Evaluation Criteria:
The post MUST meet ALL of the following:
✓ HTML Accuracy: Mentions HTML elements only if present in USER_CODE's HTML structure
✓ CSS Accuracy: Describes styling, layouts, and visual effects only if corresponding CSS rules exist in USER_CODE
✓ JavaScript Accuracy: Describes interactive features, DOM manipulation, or event handling only if corresponding JavaScript code exists in USER_CODE
✓ Responsive Claims: Describes responsive design only if CSS contains @media queries or responsive utility classes (e.g., Flexbox, Grid with flexible units)
✓ Interactivity Claims: Describes interactivity only if JS contains event listeners (addEventListener, onclick, etc.) or DOM manipulation (querySelector, getElementById, classList, etc.)
✓ Animation/Transition Claims: Mentions animations or transitions only if CSS includes transition, animation, transform, or @keyframes
✓ Technology Tags: Mentions technologies (HTML/CSS/JavaScript/frameworks) only if corresponding files contain non-empty, functional code
✓ No Invented Features: Does NOT describe features, libraries, frameworks, or APIs absent from USER_CODE, even if mentioned in QUESTION_DESCRIPTION
✓ Framework/Library Accuracy: Does NOT claim React, Vue, Bootstrap, jQuery, or any framework/library unless explicitly present in USER_CODE

Context Provided:
QUESTION_DESCRIPTION: {question_description}
USER_CODE: {html_code} {css_code} {js_code}
Generated Post: {output}

Scoring:
Score 1 if every described feature, technology, and behavior is verifiably present in USER_CODE and no unimplemented feature is mentioned
Score 0 if the post includes even one feature, technology, or behavior not present in USER_CODE

Be extremely strict. One inaccurate claim → score 0.

Examples for Reference:
Score: 1 (All features verified)
Post says: "Built a responsive navbar with hover effects and a click-to-toggle menu. 🎨"
User code: CSS has @media (max-width: 768px) {{...}} and .nav-item:hover {{ color: blue; }}; JS contains menuBtn.addEventListener('click', () => nav.classList.toggle('open')); HTML contains <nav> structure
Reason: Responsive design verified by media query; hover effects by CSS :hover; toggle functionality by JS event listener and classList manipulation

Score: 0 (Invented feature - API integration)
Post says: "Integrated search with live API results. 🔍"
User code: No fetch(), XMLHttpRequest, or API calls present
Reason: Claims API integration but USER_CODE contains no network requests

Score: 0 (Invented feature - animations)
Post says: "Added smooth animations for a polished experience. ✨"
User code: CSS has no transition, animation, transform, or @keyframes
Reason: Claims animations without any CSS animation code

Score: 0 (Wrong framework)
Post says: "Built with React components. ⚛️"
User code: Plain JavaScript with document.querySelector(), no JSX or React imports
Reason: Claims React framework but code is vanilla JavaScript

Return as JSON:
{{
    "reason": "<cite exact lines/sections from USER_CODE that justify accuracy OR point out specific mismatches between claims and code>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(
        self,
        output: str,
        question_description: str = "",
        html_code: str = "",
        css_code: str = "",
        js_code: str = "",
        **ignored_kwargs: Any,
    ):
        prompt = self.prompt_template.format(
            question_description=_safe_str(question_description),
            html_code=_safe_str(html_code),
            css_code=_safe_str(css_code),
            js_code=_safe_str(js_code),
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ CodeImplementationAccuracyMetric defined")


✅ CodeImplementationAccuracyMetric defined


# PlatformStructureComplianceMetric


In [7]:
class PlatformStructureComplianceMetric(base_metric.BaseMetric):
    """
    Scores 1 if the post adheres to required structure, formatting, and length rules for SHARING_PLATFORM.
    Scores 0 if ANY rule is violated.
    """
    def __init__(self, name: str = "platform_structure_compliance"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the post adheres to the required structure, formatting, and length rules for the SHARING_PLATFORM.

Evaluation Criteria:
The post MUST meet ALL of the following:

Structure Requirements (All Platforms):
✓ Three-Section Structure: Must contain exactly three sections in order:
  1. Main content (feature-focused descriptive text)
  2. URL line (lead-in phrase + PUBLISHED_URL)
  3. Hashtags line
✓ Main Content Formatting: Each sentence on a new line (no periods on the same line as next sentence)
✓ URL Line Format: Must include a lead-in phrase (e.g., "See it live:", "Try it here:", "Check it out:", "Explore it here:") followed by PUBLISHED_URL
✓ URL Appears Once: PUBLISHED_URL appears exactly once, only on the URL line
✓ Hashtags Line: Separate line starting with #NxtWave, containing only hashtags (no other text)
✓ No Extra Content: No introductory text, closing remarks, or content outside the three sections
✓ Plain Text Only: No markdown formatting, no code fences, no bold/italic markers

Platform-Specific Length Requirements:
✓ LinkedIn: 80–120 words (count words in Main content + URL line + hashtags line combined)
✓ Twitter/X: ≤280 characters (entire post including spaces, emojis, URL, and hashtags)

Emoji Requirements (All Platforms):
✓ Emoji Count: Exactly 3–4 emojis present in the entire post (not 2, not 5+)
✓ Emoji Placement: Emojis placed naturally within sentences, not clustered

Context Provided:
SHARING_PLATFORM: {social_media}
PUBLISHED_URL: {live_url}
Generated Post: {output}

Scoring:
Score 1 if ALL applicable rules for the platform are satisfied (structure, length, formatting, and emoji count)
Score 0 if ANY rule is violated

Examples for Reference:
Score: 1 (Compliant LinkedIn post)
Built a responsive task manager with clean UI and smooth interactions. 🎯
Users can add, delete, and mark tasks as complete.
The design adapts beautifully across all devices. 📱

Check it out: https://example.nxtwave.tech/task-manager

#NxtWave #Coding #JavaScript #CSS
Reason: 95 words total; three sections present; URL line has lead-in; 3 emojis; each sentence on new line; plain text

Score: 0 (Missing lead-in phrase)
Built a weather app with live data.
https://example.nxtwave.tech/weather

#NxtWave #JavaScript
Reason: URL line missing lead-in phrase (should be "See it live: https://...")

Score: 0 (Non-compliant Twitter/X - too long)
Created an interactive portfolio website with smooth scrolling, animated sections, responsive navigation, contact form validation, and dark mode toggle. 🌙
Try it: https://example.nxtwave.tech/portfolio 🚀
#NxtWave #WebDevelopment #JavaScript #CSS
Reason: 312 characters (exceeds 280 limit); also has 2 emojis instead of 3-4

Score: 0 (Wrong emoji count)
Built a calculator app. 🔢🎨📱✨🚀
See it here: https://example.nxtwave.tech/calc
#NxtWave #JavaScript
Reason: 5 emojis (requirement is 3-4)

Score: 0 (Wrong formatting - sentences not on new lines)
Built a quiz app with score tracking. Users can answer questions and see results. 📚

Try it: https://example.nxtwave.tech/quiz

#NxtWave #JavaScript
Reason: First two sentences on same line (should be separate lines)

Score: 0 (Missing hashtags section)
Developed a landing page with hero section and animations. 🎨
The layout is fully responsive.

Explore it: https://example.nxtwave.tech/landing
Reason: No hashtags line present

Return as JSON:
{{
    "reason": "<report word count (LinkedIn) OR character count (Twitter/X), emoji count, verify three-section structure with lead-in phrase, and note any formatting violations>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(
        self,
        output: str,
        social_media: str = "",
        live_url: str = "",
        **ignored_kwargs: Any,
    ):
        prompt = self.prompt_template.format(
            social_media=_safe_str(social_media),
            live_url=_safe_str(live_url),
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ PlatformStructureComplianceMetric defined")


✅ PlatformStructureComplianceMetric defined


# HashtagsAndToneMetric


In [8]:
class HashtagsAndToneMetric(base_metric.BaseMetric):
    """
    Scores 1 if ALL hashtag rules AND ALL tone/language rules are satisfied.
    Scores 0 if ANY rule is violated.
    """
    def __init__(self, name: str = "hashtags_and_tone"):
        self.name = name
        self.prompt_template = """
You are evaluating whether the post uses correct hashtags composition and maintains appropriate tone and language.

Evaluation Criteria:

Hashtag Requirements:
✓ Mandatory First Hashtag: Hashtags line MUST start with #NxtWave
✓ CCBP Category (1-2 required): Must include 1–2 hashtags from this set:
  #CCBP, #Coding, #DeveloperJourney, #Learning, #Programming, #NextLevelCoding
✓ Technology Tags (1-2 required): Must include 1–2 technology hashtags derived from actual code in USER_CODE:
  - Use #HTML only if HTML file contains non-empty structure
  - Use #CSS only if CSS file contains non-empty styling rules
  - Use #JavaScript only if JS file contains non-empty code
  - Use #ResponsiveDesign only if CSS contains @media queries
  - Do NOT use framework tags (#React, #Vue, #Angular) unless explicitly present in code
✓ No Duplicates: Each hashtag appears only once
✓ No Unrelated Tags: Only use hashtags from the allowed categories above
✓ Hashtags Position: All hashtags appear only on the final line (not scattered in main content)

Tone and Language Requirements:
✓ Professional Simplicity: Use professional yet simple English suitable for beginner–intermediate level
✓ Positive & Enthusiastic: Maintain positive and enthusiastic tone throughout
✓ Avoid Academic Terms: Never use words like "assignment", "task", "requirements", "submission", "homework"
✓ Project Framing: Describe the work as a "project" or use neutral terms like "built", "created", "developed"
✓ No Technical Jargon: Avoid unnecessarily complex technical terms; use clear, accessible language
✓ Active Voice: Use active voice and first-person perspective (e.g., "Built a dashboard" not "A dashboard was built")

Context Provided:
USER_CODE: {html_code} {css_code} {js_code}
Generated Post: {output}

Scoring:
Score 1 if ALL hashtag rules AND ALL tone/language rules are satisfied
Score 0 if ANY rule is violated

Examples for Reference:
Score: 1 (Compliant)
Built a modern weather dashboard with real-time updates. 🌤️
Users can search cities and view detailed forecasts.
The interface adapts seamlessly to mobile devices. 📱

See it live: https://example.nxtwave.tech/weather

#NxtWave #Coding #JavaScript #CSS
Hashtags: #NxtWave ✓, #Coding (CCBP category) ✓, #JavaScript and #CSS (from USER_CODE) ✓
Tone: Professional simple language ✓, positive tone ✓, says "Built" (not "assignment") ✓

Score: 0 (Missing #NxtWave)
Created a todo list app. ✅
Try it: https://example.nxtwave.tech/todo
#Programming #JavaScript #CSS
Reason: Hashtags line does not start with #NxtWave

Score: 0 (Wrong hashtag category count)
Developed a portfolio website with smooth scrolling. 🎨
Check it out: https://example.nxtwave.tech/portfolio
#NxtWave #HTML #CSS #JavaScript
Reason: Missing required 1-2 hashtags from CCBP category

Score: 0 (Technology not in code)
Built an interactive gallery with animations. 🖼️
Explore it: https://example.nxtwave.tech/gallery
#NxtWave #Coding #React #CSS
Reason: Uses #React but USER_CODE contains only vanilla JavaScript

Score: 0 (Used "assignment" - wrong tone)
Completed this assignment building a calculator app. 🔢
It performs basic arithmetic operations.
See it: https://example.nxtwave.tech/calc
#NxtWave #Programming #JavaScript
Reason: Uses forbidden term "assignment"

Score: 0 (Duplicate hashtag)
Created a landing page with hero section. 🚀
View it: https://example.nxtwave.tech/landing
#NxtWave #Coding #CSS #CSS #HTML
Reason: #CSS appears twice

Score: 0 (Hashtags in wrong location)
Built a quiz app #JavaScript with scoring. 📝
Users can track their progress.
Try it: https://example.nxtwave.tech/quiz
#NxtWave #Coding #HTML
Reason: #JavaScript appears in main content instead of only on hashtags line

Score: 0 (Missing CCBP category)
Developed a music player. 🎵
Listen here: https://example.nxtwave.tech/player
#NxtWave #JavaScript #HTML #CSS
Reason: Has only #NxtWave and technology tags, but missing 1-2 from CCBP category

Return as JSON:
{{
    "reason": "<list all found hashtags, classify each into: [Mandatory], [CCBP Category], or [Technology]; verify each technology hashtag against USER_CODE presence; count categories; verify tone compliance (check for forbidden words, emoji count 3-4); note all violations>",
    "score": <0 or 1>
}}
Return only valid JSON with exactly two keys: reason and score.
"""

    def score(
        self,
        output: str,
        html_code: str = "",
        css_code: str = "",
        js_code: str = "",
        **ignored_kwargs: Any,
    ):
        prompt = self.prompt_template.format(
            html_code=_safe_str(html_code),
            css_code=_safe_str(css_code),
            js_code=_safe_str(js_code),
            output=output,
        )
        result = call_evaluator(prompt)
        return score_result.ScoreResult(
            name=self.name,
            value=float(result["score"]),
            reason=result["reason"],
        )

print("✅ HashtagsAndToneMetric defined")


✅ HashtagsAndToneMetric defined


# Assemble Metrics


In [9]:
web_share_metrics = [
    CodeImplementationAccuracyMetric(),
    PlatformStructureComplianceMetric(),
    HashtagsAndToneMetric(),
]

print(f"✅ {len(web_share_metrics)} metrics ready:")
for m in web_share_metrics:
    print(f"   • {m.name}")


✅ 3 metrics ready:
   • code_implementation_accuracy
   • platform_structure_compliance
   • hashtags_and_tone


# Experiment — Run Evaluation


In [ ]:
dataset = opik_client.get_dataset(name=DATASET_NAME)
tracked_llm_client = track_openai(openrouter_client)


def llm_call(dataset_item: dict) -> str:
    try:
        prompt = build_web_share_prompt(dataset_item)
        response = tracked_llm_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.001,
        )
        content = response.choices[0].message.content
        return content if content else ""
    except Exception as e:
        return f"Error: {str(e)}"


def evaluation_task(dataset_item: dict) -> dict:
    output = llm_call(dataset_item)
    return {
        "output": output,
        "question_description": _safe_str(dataset_item.get("question_description")),
        "html_code": _safe_str(dataset_item.get("html_code")),
        "css_code": _safe_str(dataset_item.get("css_code")),
        "js_code": _safe_str(dataset_item.get("js_code")),
        "social_media": _safe_str(dataset_item.get("social_media")),
        "live_url": _safe_str(dataset_item.get("live_url")),
    }


print(f"🚀 Starting evaluation : {EXPERIMENT_NAME}")
print(f"📁 Dataset             : {DATASET_NAME}")
print(f"🤖 Model               : {MODEL_NAME}")
print(f"📊 Metrics             : {len(web_share_metrics)}")
print(f"Evaluate object : {evaluate}")

dataset.get_version_info = lambda: None

eval_results = evaluate(
    experiment_name=EXPERIMENT_NAME,
    dataset=dataset,
    task=evaluation_task,
    scoring_metrics=web_share_metrics,
    task_threads=2,
    verbose=1,
)

print("✅ Evaluation completed")


🚀 Starting evaluation : web-share-eval_google/openai/gpt-5.4-mini
📁 Dataset             : ai_eval_web_share_prod
🤖 Model               : openai/gpt-5.4-mini
📊 Metrics             : 3
Evaluate object : <function evaluate at 0x7c8616df0fe0>


Evaluation (there might be a delay before the first items are processed): 0it [00:00, ?it/s]

OPIK: Started logging traces to the "AI Evaluations" project at https://3.110.54.210/api/v1/session/redirect/projects/?trace_id=019f03ce-ec1c-7e9e-bd68-056614dd2cb2&path=aHR0cHM6Ly8zLjExMC41NC4yMTAvYXBp.
